# 4B core benchmark

Complete comparison of `no_loc`, integer-coordinate `loc_text` and L40 `loc_embed`, including matched shuffled-coordinate controls.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
evaluation_root = repo_root / "outputs" / "evaluation"

runs = {
    "no_loc": "11442",
    "loc_text integer": "11443",
    "loc_embed L40": "11444",
}
shuffled_runs = {
    "loc_text integer": "11447",
    "loc_embed L40": "11448",
}
condition_order = list(runs)

def read_json(path):
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)

def load_summary(job):
    return read_json(evaluation_root / job / "scored_predictions" / "summary.json")

summaries = {condition: load_summary(job) for condition, job in runs.items()}
shuffled_summaries = {condition: load_summary(job) for condition, job in shuffled_runs.items()}
predictions = {
    condition: pd.read_json(evaluation_root / job / "predictions.jsonl", lines=True)
    for condition, job in runs.items()
}

pd.DataFrame({
    "Condition": condition_order,
    "Evaluation job": [runs[c] for c in condition_order],
    "Samples": [len(predictions[c]) for c in condition_order],
})

,Condition,Evaluation job,Samples
0,no_loc,11442,15029
1,loc_text integer,11443,15029
2,loc_embed L40,11444,15029


## Population check

In [2]:
reference_ids = set(predictions["no_loc"]["sample_id"].astype(str))
pd.DataFrame([
    {
        "Condition": condition,
        "Rows": len(frame),
        "Unique sample IDs": frame["sample_id"].astype(str).nunique(),
        "Same IDs as no_loc": set(frame["sample_id"].astype(str)) == reference_ids,
    }
    for condition, frame in predictions.items()
])

,Condition,Rows,Unique sample IDs,Same IDs as no_loc
0,no_loc,15029,15029,True
1,loc_text integer,15029,15029,True
2,loc_embed L40,15029,15029,True


## Main results

One primary metric per task family. Average rank weights the four task families equally; lower is better.

In [3]:
def task_row(summary, task_type):
    return next(row for row in summary["by_task_type"] if row["task_type"] == task_type)

def primary_metrics(summary):
    return {
        "Caption BLEU-4": summary["captioning"]["bleu4"],
        "Binary accuracy": task_row(summary, "binary")["accuracy"],
        "MCQ accuracy": task_row(summary, "mcq")["accuracy"],
        "Bounding-box mIoU": task_row(summary, "bounding box")["miou"],
    }

main_results = pd.DataFrame([
    {"Condition": condition, **primary_metrics(summary)}
    for condition, summary in summaries.items()
]).set_index("Condition").reindex(condition_order)
metric_columns = list(main_results.columns)
main_results["Average rank"] = main_results[metric_columns].rank(ascending=False).mean(axis=1)
main_results.style.format("{:.4f}").highlight_max(
    subset=metric_columns, axis=0, props="font-weight: bold"
).highlight_min(
    subset=["Average rank"], axis=0, props="font-weight: bold"
).set_caption("Primary benchmark metrics")

,Caption BLEU-4,Binary accuracy,MCQ accuracy,Bounding-box mIoU,Average rank
Condition,,,,,
no_loc,0.4462,0.7976,0.8067,0.6095,2.0000
loc_text integer,0.4508,0.7966,0.8164,0.6108,1.5000
loc_embed L40,0.4440,0.7959,0.8186,0.6072,2.5000


## Difference from no_loc

In [4]:
delta = main_results.loc[["loc_text integer", "loc_embed L40"], metric_columns].subtract(main_results.loc["no_loc", metric_columns])
limit = delta.abs().to_numpy().max()
delta.style.format("{:+.4f}").background_gradient(cmap="RdYlGn", vmin=-limit, vmax=limit).set_caption("Location condition − no_loc")

,Caption BLEU-4,Binary accuracy,MCQ accuracy,Bounding-box mIoU
Condition,,,,
loc_text integer,+0.0046,-0.0010,+0.0097,+0.0013
loc_embed L40,-0.0023,-0.0017,+0.0119,-0.0023


## Task-wise results

In [5]:
category_rows = []
for condition, summary in summaries.items():
    for row in summary["by_task_category"]:
        category_rows.append({"Condition": condition, **row})
category_scores = pd.DataFrame(category_rows)

def category_table(task_type, metric):
    rows = category_scores[category_scores["task_type"] == task_type]
    table = rows.pivot(index="Condition", columns="task_category", values=metric)
    overall = pd.Series({
        condition: task_row(summaries[condition], task_type)[metric]
        for condition in condition_order
    }, name="Overall")
    return table.reindex(condition_order).join(overall)

display(category_table("binary", "accuracy").style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold").set_caption("Binary accuracy"))
display(category_table("mcq", "accuracy").style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold").set_caption("MCQ accuracy"))
display(category_table("bounding box", "miou").style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold").set_caption("Bounding-box mIoU"))

,adjacency,area,count,presence,Overall
Condition,,,,,
no_loc,0.755,0.888,0.824,0.776,0.798
loc_text integer,0.754,0.887,0.812,0.785,0.797
loc_embed L40,0.751,0.895,0.815,0.778,0.796


,adjacency,area,climate zone,count,country,presence,relative pos,season,Overall
Condition,,,,,,,,,
no_loc,0.708,0.730,0.966,0.602,0.971,0.842,0.808,0.956,0.807
loc_text integer,0.705,0.740,0.979,0.610,1.000,0.856,0.824,0.970,0.816
loc_embed L40,0.718,0.732,0.984,0.612,1.000,0.852,0.819,0.973,0.819


,point,reference,Overall
Condition,,,
no_loc,0.801,0.420,0.610
loc_text integer,0.800,0.423,0.611
loc_embed L40,0.803,0.413,0.607


## Shuffled-coordinate controls

Values are shuffled minus correct. Negative values mean that replacing the true coordinates hurt performance.

In [6]:
counterfactual_rows = []
for condition in shuffled_runs:
    correct = primary_metrics(summaries[condition])
    shuffled = primary_metrics(shuffled_summaries[condition])
    counterfactual_rows.append({
        "Condition": condition,
        **{metric: shuffled[metric] - correct[metric] for metric in correct},
    })
counterfactual_deltas = pd.DataFrame(counterfactual_rows).set_index("Condition")
limit = counterfactual_deltas.abs().to_numpy().max()
counterfactual_deltas.style.format("{:+.4f}").background_gradient(cmap="RdYlGn", vmin=-limit, vmax=limit).set_caption("Shuffled − correct coordinates")

,Caption BLEU-4,Binary accuracy,MCQ accuracy,Bounding-box mIoU
Condition,,,,
loc_text integer,-0.0814,-0.0117,-0.1362,-0.0027
loc_embed L40,-0.0781,-0.0264,-0.1481,-0.0076


### Direct-geography MCQs under shuffling

In [7]:
def category_accuracy(summary, category):
    return next(
        row["accuracy"]
        for row in summary["by_task_category"]
        if row["task_type"] == "mcq" and row["task_category"] == category
    )

geo_rows = []
for condition in shuffled_runs:
    for category in ["country", "climate zone", "season"]:
        correct = category_accuracy(summaries[condition], category)
        shuffled = category_accuracy(shuffled_summaries[condition], category)
        geo_rows.append({
            "Condition": condition,
            "Category": category,
            "Correct": correct,
            "Shuffled": shuffled,
            "Difference": shuffled - correct,
        })
pd.DataFrame(geo_rows).style.format({"Correct": "{:.3f}", "Shuffled": "{:.3f}", "Difference": "{:+.3f}"})

,Condition,Category,Correct,Shuffled,Difference
0,loc_text integer,country,1.000,0.308,-0.692
1,loc_text integer,climate zone,0.979,0.440,-0.539
2,loc_text integer,season,0.970,0.784,-0.186
3,loc_embed L40,country,1.000,0.304,-0.696
4,loc_embed L40,climate zone,0.984,0.422,-0.561
5,loc_embed L40,season,0.973,0.763,-0.210


## Full diagnostic tables

In [8]:
task_type_rows = []
caption_rows = []
for condition, summary in summaries.items():
    for row in summary["by_task_type"]:
        task_type_rows.append({"Condition": condition, **row})
    caption_rows.append({"Condition": condition, **summary["captioning"]})
display(pd.DataFrame(task_type_rows).sort_values(["task_type", "Condition"]).reset_index(drop=True))
display(pd.DataFrame(caption_rows).set_index("Condition").reindex(condition_order).reset_index())
display(category_scores.sort_values(["task_type", "task_category", "Condition"]).reset_index(drop=True))

,Condition,task_type,n,extraction_success,accuracy,correct,miou,acc@25,acc@50,acc@75,acc@90
0,loc_embed L40,binary,6927,1.0,0.795871,5513.0,NaN,NaN,NaN,NaN,NaN
1,loc_text integer,binary,6927,1.0,0.796593,5518.0,NaN,NaN,NaN,NaN,NaN
2,no_loc,binary,6927,1.0,0.797604,5525.0,NaN,NaN,NaN,NaN,NaN
3,loc_embed L40,bounding box,1582,1.0,NaN,NaN,0.607179,0.785082,0.694690,0.487358,0.202276
4,loc_text integer,bounding box,1582,1.0,NaN,NaN,0.610850,0.788875,0.691530,0.489254,0.218078
5,no_loc,bounding box,1582,1.0,NaN,NaN,0.609520,0.790139,0.694058,0.481037,0.206068
6,loc_embed L40,captioning,970,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,loc_text integer,captioning,970,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,no_loc,captioning,970,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,loc_embed L40,mcq,5550,1.0,0.818559,4543.0,NaN,NaN,NaN,NaN,NaN


,Condition,bleu1,bleu2,bleu3,bleu4,meteor,cider,rouge_1,rouge_2,rouge_l
0,no_loc,0.629213,0.542056,0.485522,0.446226,0.635900,1.533020,0.706335,0.553660,0.615967
1,loc_text integer,0.631635,0.545285,0.489393,0.450791,0.642905,1.622784,0.709252,0.557916,0.618343
2,loc_embed L40,0.624895,0.538470,0.482725,0.443973,0.645020,1.561981,0.710189,0.557417,0.618594


,Condition,task_type,task_category,n,extraction_success,accuracy,correct,miou,acc@25,acc@50,acc@75,acc@90
0,loc_embed L40,binary,adjacency,2857,1.0,0.751138,2146.0,NaN,NaN,NaN,NaN,NaN
1,loc_text integer,binary,adjacency,2857,1.0,0.753588,2153.0,NaN,NaN,NaN,NaN,NaN
2,no_loc,binary,adjacency,2857,1.0,0.754988,2157.0,NaN,NaN,NaN,NaN,NaN
3,loc_embed L40,binary,area,1306,1.0,0.895100,1169.0,NaN,NaN,NaN,NaN,NaN
4,loc_text integer,binary,area,1306,1.0,0.887443,1159.0,NaN,NaN,NaN,NaN,NaN
5,no_loc,binary,area,1306,1.0,0.888208,1160.0,NaN,NaN,NaN,NaN,NaN
6,loc_embed L40,binary,count,1306,1.0,0.814701,1064.0,NaN,NaN,NaN,NaN,NaN
7,loc_text integer,binary,count,1306,1.0,0.812404,1061.0,NaN,NaN,NaN,NaN,NaN
8,no_loc,binary,count,1306,1.0,0.823890,1076.0,NaN,NaN,NaN,NaN,NaN
9,loc_embed L40,binary,presence,1458,1.0,0.777778,1134.0,NaN,NaN,NaN,NaN,NaN


## Current reading

- Both location conditions improve MCQ accuracy over `no_loc`; `loc_embed` is best.
- `loc_text` has the strongest caption metrics and slightly improves bounding-box mIoU.
- Shuffling strongly damages captioning and MCQ, confirming that both models use their coordinates.